# Generate factuality questions

In this notebook we query an LLM with the annotated dataset to generate

- Question to SmolDOC document
- Answer to question

and save it to disk, such that we do not have to ask the LLM multiple times (saving time and costs).

In [ ]:
import os
import pandas as pd

path_to_questions_answers = "../data/factuality_questions_answers.pkl"

if os.path.exists(path_to_questions_answers):
    df_questions = pd.read_pickle(path_to_questions_answers)
    print(df_questions.head())
    raise FileExistsError("File already exists. You do not need to run the following cells. Feel free to ignore this exception.")
else:
    print(
        "File does not exist. Go ahead and generate questions and answers by running the next cells."
    )

In [ ]:
from src.utils import get_extended_datasets

datasets = get_extended_datasets()

example_cfg = "smoldoc__en_sw"  # We use Swahili since this config has the full 584 document translations
annotated_dataset = datasets[example_cfg]
annotated_dataset

In [ ]:
import pandas as pd

# Get only the incorrect entries
df = pd.DataFrame(annotated_dataset)
incorrect_data = df[df["factuality"] == "has_errors"][:5]
incorrect_data

In [ ]:
QUESTION_GEN_SYSTEM_PROMPT = """
You are an expert dataset creator for factuality evaluation.
Your task is to read an English document (from the SmolDoc dataset) and produce a small set of factual question, answer pairs that test a model's factual understanding of the text.

Your goals:
1. Create one or more (up to three) concise, factual, and self-contained questions based on the given document. Only make one question per issue. Only if an issue from an annotator is semantically different to another annotator's issue, make an extra question about it.
2. Each question must have one short, unambiguous gold answer that is explicitly supported by the text.
3. Questions should be neither trivial nor adversarial — they should test meaningful factual comprehension, not obscure details or wordplay.
4. Do not explicitly refer to the source document, annotators or the issues in your question. The recipient of the question will only see the source document and then be asked a question about it. Refer only to the CONTENT of the document and base your question on the issues specified by the annotators.

Output only valid JSON, following this structure:

[
  {
    "question": "<English question>",
    "answer": "<short correct English answer>"
  }
]

- Limit each question to less than 25 words.
- Limit each answer to less than 10 words.
"""

In [ ]:
from src.llm_chat import AzureOpenAIChatter, CachedLLMChat, LLMChat

# chatter = OllamaChatter(model_name="deepseek-r1:8b", think=True)
chatter = AzureOpenAIChatter()
chat = LLMChat(chatter)
chat = CachedLLMChat(chat, cache_file_path="../data/factuality_question_gen_cache.pkl")

questions_with_answers: list[dict[str, str]] = []


def parse_response(response: str, id: str):
    """
    Parse the LLM response as JSON and attach topic_id.

    Args:
        response (str): The LLM response string.
        id (str): The topic ID to attach.
    Returns:
        list[dict] | None: The parsed JSON with topic_id added, or None on failure.
    """
    try:
        import json

        json_data = response
        loaded_json = json.loads(json_data)
        for entry in loaded_json:
            entry["topic_id"] = id
        return loaded_json
    except Exception as e:
        print(f"Error parsing response for id {id}: {e}")
        print(f"Response was: {response}")
        return None

In [ ]:
from tqdm.notebook import tqdm

for idx, row in tqdm(incorrect_data.iterrows(), total=len(incorrect_data), desc="Generating questions"):
    id = row["id"]
    srcs = " ".join(row["srcs"])
    errors = "\n\n".join(
        (row["annotator_1_notes"], row["annotator_2_notes"], row["annotator_3_notes"])
    )
    chat.add_message("system", QUESTION_GEN_SYSTEM_PROMPT)
    response, thoughts = chat.chat(
        f"""Here is the source document: {srcs}\n
        Annotators have noted the following issues:
        {errors}\n
        Generate question and answers in the specified JSON format that adheres to the goals and limitations given."""
    )
    parsed_json = parse_response(response, id)
    questions_dicts = parsed_json or []
    for dict in questions_dicts: # Add debugging information
        dict["reasoning"] = thoughts
        dict["source_document"] = row["srcs"]
        dict["annotator_1_notes"] = row["annotator_1_notes"]
        dict["annotator_2_notes"] = row["annotator_2_notes"]
        dict["annotator_3_notes"] = row["annotator_3_notes"]

    questions_with_answers.extend(questions_dicts)
    chat.reset()

In [ ]:
df_questions = pd.DataFrame(questions_with_answers)
df_questions = df_questions[["topic_id", "question", "answer", "source_document", "annotator_1_notes", "annotator_2_notes", "annotator_3_notes", "reasoning"]] # Change the order of columns
df_questions

In [ ]:
# save to file
df_questions.to_pickle(path_to_questions_answers)